# Brownian glued-bulk validation

This notebook checks whether finite glued-bulk buffers give the same Brownian front velocity as the no-glue reference. The no-glue reference is implemented with `B = 1e6`, using the same glued C code.

## Imports

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

from front_runner import compile_c_code, run_front_simulations

try:
    from scipy.stats import ttest_ind
except Exception:
    ttest_ind = None

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 13,
    "axes.labelsize": 25,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

def style_axis(ax, n=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=n))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=n))
    ax.grid(False)

def sem(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) <= 1:
        return np.nan
    return np.std(values, ddof=1) / np.sqrt(len(values))

## Workflow settings

In [ ]:
project_folder = Path('.').resolve()

c_file = project_folder / 'abm_front_propagation_glued.c'
exe_file = project_folder / 'abm_front_glued_validation_brownian.exe'

param_base = 'params_validation_brownian_B_sweep'
param_file = project_folder / f'{param_base}.txt'
run_table_file = project_folder / f'{param_base}_runs.csv'

data_folder = project_folder / 'data' / param_base
figure_folder = project_folder / 'figures' / param_base
summary_folder = project_folder / 'analysis_summaries' / param_base

for folder in [data_folder, figure_folder, summary_folder]:
    folder.mkdir(parents=True, exist_ok=True)

compile_first = False
run_simulations = False
skip_existing_outputs = False
selected_run_ids = None     # Example: [0, 1, 2] for testing only a few runs
max_parallel = 9

print('Project folder:', project_folder)
print('C file:', c_file)
print('Executable:', exe_file)
print('Parameter file:', param_file)
print('Run table:', run_table_file)
print('Data folder:', data_folder)
print('Figure folder:', figure_folder)
print('Summary folder:', summary_folder)

if not c_file.exists():
    raise FileNotFoundError(f'Cannot find {c_file}. Put abm_front_propagation_glued.c in this folder.')

## Sweep and base parameters

In [ ]:
B_values = [1e6, 1.0, 5.0, 10.0, 30.0, 50.0]
seed_values = [69069, 123456789, 15015015]

main_method = "th_2"
main_fit_window = (0.70, 0.95)
fit_windows = [(0.60, 0.90), (0.70, 0.90), (0.70, 0.95), (0.80, 0.95)]

base = {
    "run_id": 0,
    "seed": 69069,
    "p0": 0.85,
    "q0": 0.15,
    "Ns": 50,
    "R_inter": 0.05,
    "rho0": 750.0,
    "save_per_step": 1000000,
    "front_per_step": 200,
    "rho_profile_every_front": 0,
    "isolation_buffer_factor": 30.0,
    "v0": 0.0,
    "Dr": 0.008,
    "Dtheta": 0.002,
    "dt": 0.001,
    "T": 600.0,
    "Lx": 40.0,
    "Ly": 0.5,
    "x_init_min": 19.75,
    "x_init_max": 20.25,
    "warmup_T": 200.0,
    "rho_sat": -1.0,
    "nbins_x": 800,
    "threshold_frac1": 0.2,
    "threshold_frac2": 0.5,
    "threshold_frac3": 0.8,
}

r_edge = base["p0"] - base["q0"]
initial_area = (base["x_init_max"] - base["x_init_min"]) * base["Ly"]
initial_N0 = int(base["rho0"] * initial_area + 0.5)

print(f"r = p0 - q0 = {r_edge:g}")
print(f"Lx = {base['Lx']:g}, Ly = {base['Ly']:g}, R_inter = {base['R_inter']:g}")
print(f"dx_bin = {base['Lx']/base['nbins_x']:g}")
print(f"x_init = [{base['x_init_min']:g}, {base['x_init_max']:g}], N0 ≈ {initial_N0}")
print(f"Brownian FKPP reference speed = {2*np.sqrt(base['Dr']*r_edge):.8g}")


dt_check = pd.DataFrame([
    ('v0*dt <= 0.1*R', base['v0'] * base['dt'], 0.1 * base['R_inter']),
    ('sqrt(2*Dr*dt) <= 0.1*R', np.sqrt(2.0 * base['Dr'] * base['dt']), 0.1 * base['R_inter']),
    ('sqrt(2*Dtheta*dt) <= 0.1', np.sqrt(2.0 * max(Dtheta_values) * base['dt']) if 'Dtheta_values' in globals() else np.sqrt(2.0 * base['Dtheta'] * base['dt']), 0.1),
    ('p0*dt <= 0.01', base['p0'] * base['dt'], 0.01),
    ('q0*dt <= 0.01', base['q0'] * base['dt'], 0.01),
], columns=['condition', 'value', 'limit'])
dt_check['ok'] = dt_check['value'] <= dt_check['limit']
display(dt_check)
if not bool(dt_check['ok'].all()):
    raise ValueError('At least one C time-step restriction is violated.')

## Helper functions

In [ ]:
def _try_number(value):
    try:
        if any(ch in value.lower() for ch in [".", "e"]):
            return float(value)
        return int(value)
    except Exception:
        try:
            return float(value)
        except Exception:
            return value

def _parse_metadata_pairs(parts):
    metadata = {}
    j = 0
    while j + 1 < len(parts):
        metadata[parts[j]] = _try_number(parts[j + 1])
        j += 2
    return metadata

def read_front_file(filename):
    metadata = {}
    column_names = None
    rows = []
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if line == "":
                continue
            if line.startswith("#"):
                parts = line[1:].split()
                if len(parts) > 0 and parts[0] == "step":
                    column_names = parts
                else:
                    metadata.update(_parse_metadata_pairs(parts))
                continue
            rows.append([float(x) for x in line.split()])
    if column_names is None:
        raise ValueError(f"Could not find front-file header in {filename}")
    arr = np.array(rows, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    data = {name: arr[:, j] for j, name in enumerate(column_names)}
    data["column_names"] = column_names
    return metadata, data

def read_glued_bulk_file(filename):
    column_names = None
    rows = []
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if line == "":
                continue
            if line.startswith("#"):
                parts = line[1:].split()
                if len(parts) > 0 and parts[0] == "step":
                    column_names = parts
                continue
            rows.append([float(x) for x in line.split()])
    if column_names is None:
        raise ValueError(f"Could not find glued-bulk header in {filename}")
    df = pd.DataFrame(rows, columns=column_names)
    for col in ["step", "N", "has_region", "N_active", "N_removed_previous_update"]:
        if col in df:
            df[col] = df[col].astype(int)
    return df

threshold_methods = ["th_1", "th_2", "th_3"]

def threshold_fraction(method, metadata):
    index = int(method.split("_")[1])
    return float(metadata.get(f"threshold_frac{index}", np.nan))

def method_label(method, metadata):
    frac = threshold_fraction(method, metadata)
    if np.isfinite(frac):
        return fr"{method} ($\alpha={frac:g}$)"
    return method

def front_columns(side, method):
    if method == "tip":
        return "x_left_tip" if side == "left" else "x_right_tip"
    if method == "quantile":
        return "x_q01" if side == "left" else "x_q99"
    if method in threshold_methods:
        k = method.split("_")[1]
        return f"x_{side}_th_{k}"
    raise ValueError(f"Unknown front method: {method}")

def fit_one_side(front_data, side, method, fit_window):
    time = np.asarray(front_data["time"], dtype=float)
    t_end = np.nanmax(time)
    t_min = fit_window[0] * t_end
    t_max = fit_window[1] * t_end
    column = front_columns(side, method)
    x = np.asarray(front_data[column], dtype=float)
    mask = np.isfinite(time) & np.isfinite(x) & (time >= t_min) & (time <= t_max)
    if np.sum(mask) < 2:
        return {
            "slope": np.nan, "intercept": np.nan, "speed": np.nan,
            "mse": np.nan, "t_end": t_end, "t_fit_min": t_min,
            "t_fit_max": t_max, "n_fit": int(np.sum(mask)),
        }
    slope, intercept = np.polyfit(time[mask], x[mask], 1)
    prediction = slope * time[mask] + intercept
    mse = float(np.mean((x[mask] - prediction) ** 2))
    speed = -slope if side == "left" else slope
    return {
        "slope": float(slope), "intercept": float(intercept),
        "speed": float(speed), "mse": mse, "t_end": float(t_end),
        "t_fit_min": float(t_min), "t_fit_max": float(t_max),
        "n_fit": int(np.sum(mask)),
    }

def fit_front_speed(front_data, method="th_2", fit_window=(0.70, 0.95)):
    left = fit_one_side(front_data, "left", method, fit_window)
    right = fit_one_side(front_data, "right", method, fit_window)
    return {
        "method": method,
        "fit_window": f"{fit_window[0]:.2f}-{fit_window[1]:.2f}",
        "left_speed": left["speed"],
        "right_speed": right["speed"],
        "v_front": np.nanmean([left["speed"], right["speed"]]),
        "left_mse": left["mse"],
        "right_mse": right["mse"],
        "mean_mse": np.nanmean([left["mse"], right["mse"]]),
        "left_intercept": left["intercept"],
        "right_intercept": right["intercept"],
        "asymmetry_left_minus_right": left["speed"] - right["speed"],
        "t_end": left["t_end"],
        "t_fit_min": left["t_fit_min"],
        "t_fit_max": left["t_fit_max"],
        "n_fit": min(left["n_fit"], right["n_fit"]),
    }

def glue_diagnostics(glued_file, t_fit_min, t_fit_max, no_glue=False):
    if no_glue:
        return {
            "first_glue_time": np.nan,
            "fit_glued_fraction": np.nan,
            "glue_active_before_fit": np.nan,
            "glue_note": "no-glue reference",
        }
    glued_file = Path(glued_file)
    if not glued_file.exists():
        return {
            "first_glue_time": np.nan,
            "fit_glued_fraction": np.nan,
            "glue_active_before_fit": False,
            "glue_note": "missing glued-bulk file",
        }
    bulk = read_glued_bulk_file(glued_file)
    has = bulk["has_region"].to_numpy(dtype=bool)
    time = bulk["time"].to_numpy(dtype=float)
    first_time = np.nan
    if np.any(has):
        first_time = float(time[has][0])
    in_fit = (time >= t_fit_min) & (time <= t_fit_max)
    fit_fraction = np.nan
    if np.any(in_fit):
        fit_fraction = float(np.mean(has[in_fit]))
    ok = bool(np.isfinite(first_time) and first_time <= t_fit_min and fit_fraction >= 0.95)
    note = "ok" if ok else "check activation"
    return {
        "first_glue_time": first_time,
        "fit_glued_fraction": fit_fraction,
        "glue_active_before_fit": ok,
        "glue_note": note,
    }

def format_B(B):
    B = float(B)
    if B >= 1e5:
        return "no glue"
    return f"{B:g}"

def expected_files(data_folder, param_base, run_id):
    data_folder = Path(data_folder)
    return {
        "front": data_folder / f"front_{param_base}_run_{run_id:03d}.dat",
        "glued": data_folder / f"glued_bulk_{param_base}_run_{run_id:03d}.dat",
        "snapshot": data_folder / f"snapshot_{param_base}_run_{run_id:03d}.dat",
    }

def load_validation_results(runs_df, data_folder, param_base,
                            main_method="th_2",
                            main_fit_window=(0.70, 0.95),
                            fit_windows=None,
                            group_columns=None):
    if fit_windows is None:
        fit_windows = [main_fit_window]
    if group_columns is None:
        group_columns = []

    main_records = []
    window_records = []
    threshold_records = []
    missing = []

    for _, run in runs_df.iterrows():
        run_id = int(run["run_id"])
        files = expected_files(data_folder, param_base, run_id)
        if not files["front"].exists():
            missing.append(run_id)
            continue

        metadata, front_data = read_front_file(files["front"])
        base_info = {col: run[col] for col in runs_df.columns}
        base_info["B_label"] = format_B(run["isolation_buffer_factor"])

        main_fit = fit_front_speed(front_data, method=main_method, fit_window=main_fit_window)
        glue = glue_diagnostics(
            files["glued"],
            main_fit["t_fit_min"],
            main_fit["t_fit_max"],
            no_glue=float(run["isolation_buffer_factor"]) >= 1e5,
        )
        main_records.append({**base_info, **main_fit, **glue})

        for fw in fit_windows:
            fit = fit_front_speed(front_data, method=main_method, fit_window=fw)
            window_records.append({**base_info, **fit})

        for method in threshold_methods:
            fit = fit_front_speed(front_data, method=method, fit_window=main_fit_window)
            fit["threshold_fraction"] = threshold_fraction(method, metadata)
            threshold_records.append({**base_info, **fit})

    if missing:
        print("Missing front files for run_id:", missing)

    summary = pd.DataFrame(main_records)
    fit_window_summary = pd.DataFrame(window_records)
    threshold_summary = pd.DataFrame(threshold_records)
    return summary, fit_window_summary, threshold_summary

def summarize_velocity(summary, group_columns=None):
    if group_columns is None:
        group_columns = []
    rows = []
    for keys, sub in summary.groupby(group_columns + ["isolation_buffer_factor"], dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_columns + ["isolation_buffer_factor"], keys))
        vals = sub["v_front"].to_numpy(dtype=float)
        row.update({
            "B": row.pop("isolation_buffer_factor"),
            "B_label": format_B(keys[-1]),
            "n": int(np.isfinite(vals).sum()),
            "mean_v": float(np.nanmean(vals)),
            "std_v": float(np.nanstd(vals, ddof=1)) if np.isfinite(vals).sum() > 1 else np.nan,
            "sem_v": sem(vals),
            "mean_first_glue_time": float(np.nanmean(sub["first_glue_time"])) if "first_glue_time" in sub else np.nan,
            "min_fit_glued_fraction": float(np.nanmin(sub["fit_glued_fraction"])) if "fit_glued_fraction" in sub and np.isfinite(sub["fit_glued_fraction"]).any() else np.nan,
            "all_glue_active_before_fit": bool(sub["glue_active_before_fit"].dropna().all()) if "glue_active_before_fit" in sub and len(sub["glue_active_before_fit"].dropna()) else np.nan,
        })
        rows.append(row)
    return pd.DataFrame(rows)

def welch_p_value(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) < 2 or len(b) < 2 or ttest_ind is None:
        return np.nan
    return float(ttest_ind(a, b, equal_var=False).pvalue)

def compare_to_reference(summary, group_columns=None, reference_B=1e6):
    if group_columns is None:
        group_columns = []
    rows = []
    grouped = summary.groupby(group_columns, dropna=False) if group_columns else [((), summary)]

    for keys, sub_all in grouped:
        if group_columns and not isinstance(keys, tuple):
            keys = (keys,)
        key_info = dict(zip(group_columns, keys if group_columns else []))
        ref = sub_all[np.isclose(sub_all["isolation_buffer_factor"], reference_B)]
        if len(ref) == 0:
            continue
        ref_values = ref["v_front"].to_numpy(dtype=float)
        ref_mean = float(np.nanmean(ref_values))
        ref_sem = sem(ref_values)

        for B, sub in sub_all.groupby("isolation_buffer_factor"):
            values = sub["v_front"].to_numpy(dtype=float)
            mean_v = float(np.nanmean(values))
            sem_v = sem(values)
            diff = mean_v - ref_mean
            sem_diff = np.sqrt(sem_v**2 + ref_sem**2) if np.isfinite(sem_v) and np.isfinite(ref_sem) else np.nan
            d_sem = diff / sem_diff if np.isfinite(sem_diff) and sem_diff > 0 else np.nan
            rel_diff = 100.0 * diff / ref_mean if ref_mean != 0 else np.nan
            rel_unc = 100.0 * sem_diff / ref_mean if ref_mean != 0 and np.isfinite(sem_diff) else np.nan

            row = {
                **key_info,
                "B": float(B),
                "B_label": format_B(B),
                "n": int(np.isfinite(values).sum()),
                "mean_v": mean_v,
                "sem_v": sem_v,
                "reference_mean_v": ref_mean,
                "reference_sem_v": ref_sem,
                "diff_v": diff,
                "rel_diff_percent": rel_diff,
                "rel_diff_uncertainty_percent": rel_unc,
                "d_sem": d_sem,
                # Backward-compatible alias for older analysis cells.
                "z_diff": d_sem,
                "welch_p": welch_p_value(values, ref_values),
                "mean_first_glue_time": float(np.nanmean(sub["first_glue_time"])) if np.isfinite(sub["first_glue_time"]).any() else np.nan,
                "min_fit_glued_fraction": float(np.nanmin(sub["fit_glued_fraction"])) if np.isfinite(sub["fit_glued_fraction"]).any() else np.nan,
                "all_glue_active_before_fit": bool(sub["glue_active_before_fit"].dropna().all()) if len(sub["glue_active_before_fit"].dropna()) else np.nan,
            }
            rows.append(row)

    out = pd.DataFrame(rows)
    if len(out) > 0:
        finite = out["B"] < 1e5
        out["rank_by_abs_rel_diff"] = np.nan
        out.loc[finite, "rank_by_abs_rel_diff"] = out.loc[finite, "rel_diff_percent"].abs().rank(method="min")
    return out

## Build run table and parameter file

In [ ]:
param_order = [
    'run_id', 'seed',
    'p0', 'q0', 'Ns', 'R_inter', 'rho0',
    'save_per_step', 'front_per_step', 'rho_profile_every_front',
    'isolation_buffer_factor',
    'v0', 'Dr', 'Dtheta', 'dt', 'T',
    'Lx', 'Ly', 'x_init_min', 'x_init_max', 'warmup_T',
    'rho_sat', 'nbins_x', 'threshold_frac1', 'threshold_frac2', 'threshold_frac3',
]

runs = []
run_id = 0
for B in B_values:
    for seed in seed_values:
        run = dict(base)
        run.update(
            run_id=run_id,
            seed=int(seed),
            isolation_buffer_factor=float(B),
        )
        run['B'] = run['isolation_buffer_factor']
        run['B_label'] = format_B(run['B'])
        run['isolation_buffer'] = run['isolation_buffer_factor'] * run['R_inter']
        run['r_edge'] = run['p0'] - run['q0']
        run['v_fkpp_brownian'] = 2.0 * np.sqrt(run['Dr'] * run['r_edge'])
        run['code_type'] = 'glued'
        runs.append(run)
        run_id += 1

run_table = pd.DataFrame(runs)
run_table.to_csv(run_table_file, index=False)
runs_df = run_table.copy()

with open(param_file, 'w') as f:
    f.write('# Brownian glued-bulk validation sweep.\n')
    f.write('# No-glue reference is B = 1e6, using the same glued C code.\n')
    f.write('# Finite B values test whether the glued-bulk method changes the measured velocity.\n')
    f.write(f'# B values = {list(map(float, B_values))}\n')
    f.write(f'# seeds = {list(map(int, seed_values))}\n')
    f.write(f'# Lx = {base["Lx"]:.16g}, Ly = {base["Ly"]:.16g}, R_inter = {base["R_inter"]:.16g}\n')
    f.write(f'# T = {base["T"]:.16g}, dt = {base["dt"]:.16g}\n')
    f.write('# Fit windows are applied later during analysis, not in the C simulation.\n\n')
    for run in runs:
        f.write('[run]\n')
        for key in param_order:
            f.write(f'{key} = {run[key]}\n')
        f.write('\n')

print('Wrote:', param_file)
print('Wrote:', run_table_file)
display(run_table[['run_id', 'seed', 'B_label', 'v0', 'Dr', 'Dtheta', 'T', 'Lx', 'Ly', 'R_inter']].head(12))
print(f'Total runs: {len(run_table)}')

## Compile and run

In [ ]:
if compile_first:
    compile_c_code(c_file, exe_file)

if run_simulations:
    run_results = run_front_simulations(
        exe_file=exe_file,
        param_file=param_file,
        max_parallel=max_parallel,
        selected_run_ids=selected_run_ids,
        data_folder=data_folder,
        skip_existing=skip_existing_outputs,
        required_output='front',
    )
    display(pd.DataFrame(run_results))
else:
    print('run_simulations = False; using existing output files.')

## Analysis

In [ ]:
summary, fit_window_summary, threshold_summary = load_validation_results(
    runs_df,
    data_folder=data_folder,
    param_base=param_base,
    main_method=main_method,
    main_fit_window=main_fit_window,
    fit_windows=fit_windows,
)

velocity_stats = summarize_velocity(summary)
validation_table = compare_to_reference(summary)

finite_B_order = [1.0, 5.0, 10.0, 30.0, 50.0]
B_plot_order = finite_B_order + [1e6]

# Keep the run-level output for reproducibility, but avoid printing
# unnecessary intermediate statistical quantities.
summary.to_csv(
    summary_folder / "brownian_B_run_summary.csv",
    index=False,
)

display(
    summary[
        [
            "run_id",
            "seed",
            "B_label",
            "v_front",
            "first_glue_time",
            "fit_glued_fraction",
            "glue_active_before_fit",
        ]
    ]
)

print(f"Saved run summary in {summary_folder}")

## Diagnostics

In [ ]:
# Fit-window sensitivity.
window_stats = (
    fit_window_summary
    .groupby(["B", "B_label", "fit_window"])["v_front"]
    .agg(mean="mean", sem=sem, n="count")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for B in B_plot_order:
    sub = window_stats[np.isclose(window_stats["B"], B)]
    if len(sub) == 0:
        continue
    sub = sub.set_index("fit_window").loc[[f"{a:.2f}-{b:.2f}" for a, b in fit_windows]].reset_index()
    ax.errorbar(sub["fit_window"], sub["mean"], yerr=sub["sem"], marker="o", linewidth=1.4, capsize=3, label=format_B(B))
ax.set_xlabel("fit window")
ax.set_ylabel(r"$v_{front}$")
ax.legend(title="B", fontsize=9)
style_axis(ax, n=4)
fig.tight_layout()
fig.savefig(figure_folder / "diagnostic_fit_window_sensitivity.png", bbox_inches="tight")
plt.show()

# Threshold robustness in the main fit window.
th_stats = (
    threshold_summary
    .groupby(["B", "B_label", "method", "threshold_fraction"])["v_front"]
    .agg(mean="mean", sem=sem, n="count")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7.2, 4.2))
for B in B_plot_order:
    sub = th_stats[np.isclose(th_stats["B"], B)].sort_values("threshold_fraction")
    if len(sub) == 0:
        continue
    ax.errorbar(sub["threshold_fraction"], sub["mean"], yerr=sub["sem"], marker="o", linewidth=1.4, capsize=3, label=format_B(B))
ax.set_xlabel(r"threshold fraction $\alpha$")
ax.set_ylabel(r"$v_{front}$")
ax.legend(title="B", fontsize=9)
style_axis(ax, n=4)
fig.tight_layout()
fig.savefig(figure_folder / "diagnostic_threshold_robustness.png", bbox_inches="tight")
plt.show()

# Left-right asymmetry.
asym_stats = (
    summary
    .groupby(["B", "B_label"])["asymmetry_left_minus_right"]
    .agg(mean="mean", sem=sem, n="count")
    .reset_index()
)
asym_stats["order"] = asym_stats["B"].map({B: i for i, B in enumerate(B_plot_order)})
asym_stats = asym_stats.sort_values("order")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.errorbar(asym_stats["B_label"], asym_stats["mean"], yerr=asym_stats["sem"], marker="o", linewidth=1.4, capsize=3)
ax.axhline(0.0, linestyle="--", linewidth=1.0)
ax.set_xlabel("B")
ax.set_ylabel(r"$v_L - v_R$")
style_axis(ax, n=4)
fig.tight_layout()
fig.savefig(figure_folder / "diagnostic_left_right_asymmetry.png", bbox_inches="tight")
plt.show()

## Final validation: paired $t$-test

Each finite-$B$ simulation has a no-glue simulation with the same seed
and initial realization. The appropriate comparison is therefore a
two-sided paired $t$-test applied to the three seed-by-seed velocity
differences,

$$
\Delta v_i(B)=v_i(B)-v_i^{\mathrm{no\ glue}}.
$$

For each $B$,

$$
H_0:\langle \Delta v(B)\rangle=0,
\qquad
H_1:\langle \Delta v(B)\rangle\neq 0.
$$

We use $\alpha=0.05$. When $p\geq0.05$, the test does not detect a
statistically significant difference between the two implementations.
This does not prove equality, especially with only three pairs, so the
relative difference in the mean velocity is shown together with the
test result.

In [ ]:
from scipy.stats import ttest_rel

alpha = 0.05

# ==========================================================
# PAIRED T-TEST TABLE
# ==========================================================

reference = (
    summary[
        np.isclose(
            summary["isolation_buffer_factor"],
            1e6,
        )
    ][["seed", "v_front"]]
    .rename(columns={"v_front": "v_no_glue"})
)

test_rows = []

for B in finite_B_order:
    glued = (
        summary[
            np.isclose(
                summary["isolation_buffer_factor"],
                B,
            )
        ][["seed", "v_front"]]
        .rename(columns={"v_front": "v_glued"})
    )

    paired = (
        glued
        .merge(reference, on="seed", how="inner")
        .dropna()
    )

    result = ttest_rel(
        paired["v_glued"],
        paired["v_no_glue"],
    )

    p_value = float(result.pvalue)
    decision = (
        "Difference detected"
        if p_value < alpha
        else "No significant difference detected"
    )

    test_rows.append({
        "B": B,
        "n_pairs": len(paired),
        "p_value": p_value,
        "decision": decision,
    })

test_table = pd.DataFrame(test_rows)

display(test_table)

test_table.to_csv(
    summary_folder / "brownian_paired_t_test.csv",
    index=False,
)

test_table.to_latex(
    summary_folder / "brownian_paired_t_test.tex",
    index=False,
    float_format="%.4g",
)

# ==========================================================
# FINAL FIGURE:
# VELOCITY AND RELATIVE DIFFERENCE
# ==========================================================

finite_B_order = sorted(
    velocity_stats.loc[
        velocity_stats["B"] < 1e5,
        "B",
    ].unique()
)
B_plot_order = finite_B_order + [1e6]

x_positions = np.arange(len(B_plot_order))
x_labels = [
    f"{B:g}" if B < 1e5 else "no glue"
    for B in B_plot_order
]

vel_plot = velocity_stats.copy()
vel_plot["order"] = vel_plot["B"].map(
    {B: i for i, B in enumerate(B_plot_order)}
)
vel_plot = vel_plot.sort_values("order")

ref_row = vel_plot[
    np.isclose(vel_plot["B"], 1e6)
].iloc[0]

comp = validation_table[
    validation_table["B"] < 1e5
].copy()

comp["order"] = comp["B"].map(
    {B: i for i, B in enumerate(B_plot_order)}
)

comp["relative_difference_percent"] = (
    100.0
    * (
        comp["mean_v"]
        - comp["reference_mean_v"]
    )
    / comp["reference_mean_v"]
)

comp = comp.sort_values("order")

fig, axes = plt.subplots(
    2,
    1,
    figsize=(6.8, 7.0),
    sharex=True,
    gridspec_kw={"height_ratios": [1.15, 1.0]},
)

# Mean velocity and SEM.
axes[0].errorbar(
    vel_plot["order"],
    vel_plot["mean_v"],
    yerr=vel_plot["sem_v"],
    marker="o",
    linewidth=1.6,
    capsize=4,
)

axes[0].axhline(
    ref_row["mean_v"],
    linestyle="--",
    linewidth=1.2,
    label="no-glue mean",
)

if np.isfinite(ref_row["sem_v"]):
    axes[0].axhspan(
        ref_row["mean_v"] - ref_row["sem_v"],
        ref_row["mean_v"] + ref_row["sem_v"],
        alpha=0.15,
        label="no-glue SEM",
    )

axes[0].set_ylabel(r"$v_{\mathrm{front}}$")
axes[0].legend(frameon=False, fontsize=15)
style_axis(axes[0], n=4)

# Relative difference in the mean velocity.
axes[1].bar(
    comp["order"],
    comp["relative_difference_percent"],
    width=0.55,
)

axes[1].axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

axes[1].set_xlabel(r"glued-bulk buffer $B$")
axes[1].set_ylabel(
    r"$\Delta v_{\mathrm{rel}}(B)$"
)
style_axis(axes[1], n=4)

for ax in axes:
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels)
    ax.set_xlim(-0.6, len(B_plot_order) - 0.4)

axes[0].tick_params(axis="x", labelbottom=False)

fig.tight_layout()

figure_file = (
    figure_folder
    / "thesis_brownian_velocity_and_relative_difference_vs_B.pdf"
)
fig.savefig(
    figure_file,
    bbox_inches="tight",
)

print("Saved:", figure_file)
print(
    "Interpretation: p >= 0.05 means that the paired test "
    "does not detect a significant difference."
)

plt.show()